# TAMU API Quickstart for Colab

This notebook is self-contained. It does not import anything from this repo, and it installs its own Python dependency before connecting to the TAMU-backed Azure OpenAI endpoint.

What this notebook does:
- installs `openai`
- loads config from environment variables, Colab Secrets, or a local `.env` file if one exists
- shows a masked configuration summary
- makes one tiny API call to prove the connection works
- gives you a copy-paste example for future use

What the key is:
- `TAMU_API_KEY` is your secret authentication token
- it is not the endpoint URL
- it is not the model name
- do not paste the raw key into screenshots or commits


In [1]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_spec: str) -> None:
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])
    else:
        print(f"{import_name} already available.")

ensure_package("openai", "openai>=1.54.0")

import openai
print("openai version:", openai.__version__)


openai already available.


openai version: 1.54.3


## Configuration Sources

This notebook checks these sources in order:
1. environment variables already set in the runtime
2. Colab Secrets named `TAMU_API_KEY`, `TAMU_AZURE_ENDPOINT`, `TAMU_API_VERSION`, `OPENAI_MODEL`, `TAMU_DEPLOYMENT`
3. a local `.env` file if one exists next to the notebook or in the current working directory

If you are testing in Colab, the cleanest setup is usually Colab Secrets or setting the values in a code cell before the connection test.


In [2]:
from pathlib import Path
import os
from urllib.parse import urlparse


def load_env_file_if_present() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
        Path(".env"),
    ]
    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved in seen or not resolved.is_file():
            continue
        seen.add(resolved)
        for raw_line in resolved.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export "):].strip()
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {'"', "'"}:
                value = value[1:-1]
            os.environ.setdefault(key, value)
        return resolved
    return None


def load_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return ""
    try:
        value = userdata.get(name)
    except Exception:
        return ""
    return (value or "").strip()


def pick_value(*names: str, default: str = "") -> str:
    for name in names:
        value = os.getenv(name, "").strip()
        if value:
            return value
    for name in names:
        value = load_colab_secret(name)
        if value:
            return value
    return default


def mask_secret(secret: str) -> str:
    if not secret:
        return "<missing>"
    return f"<present: {len(secret)} chars>"


loaded_env_path = load_env_file_if_present()
deployment_aliases = {
    "protected.gpt-5.2": "gpt-5.2-deep-learning-fundamentals",
    "gpt-5.2": "gpt-5.2-deep-learning-fundamentals",
}

api_key = pick_value("TAMU_API_KEY", "OPENAI_API_KEY", "TAMUS_AI_CHAT_API_KEY")
azure_endpoint = pick_value("TAMU_AZURE_ENDPOINT", "AZURE_OPENAI_ENDPOINT", "TAMU_API_ENDPOINT", "DPF_URL")
api_version = pick_value("TAMU_API_VERSION", "AZURE_OPENAI_API_VERSION", "OPENAI_API_VERSION", default="2024-12-01-preview")
requested_model = pick_value("OPENAI_MODEL", default="protected.gpt-5.2")
deployment = pick_value("TAMU_DEPLOYMENT", "TAMU_DEPLOYMENT_NAME", "AZURE_OPENAI_DEPLOYMENT", "AZURE_OPENAI_DEPLOYMENT_NAME")
deployment = deployment or deployment_aliases.get(requested_model, requested_model)

print("Loaded .env from        :", loaded_env_path or "<none>")
print("TAMU_API_KEY            :", mask_secret(api_key))
print("TAMU_AZURE_ENDPOINT host:", urlparse(azure_endpoint).netloc or "<missing>")
print("TAMU_API_VERSION        :", api_version)
print("OPENAI_MODEL            :", requested_model)
print("Resolved deployment     :", deployment)


Loaded .env from        : C:\Users\pipin\Desktop\Coding\Personal\2026\CodeBoost\LLM_PV\.env
TAMU_API_KEY            : <present: 84 chars>
TAMU_AZURE_ENDPOINT host: tamu-it-ae-ai-prod-prod-eastus2.openai.azure.com
TAMU_API_VERSION        : 2024-12-01-preview
OPENAI_MODEL            : protected.gpt-5.2
Resolved deployment     : gpt-5.2-deep-learning-fundamentals


## Optional Manual Override

If the summary above shows missing values, uncomment the lines below and paste your TAMU values directly into this notebook runtime.


In [3]:
from getpass import getpass

# Uncomment these lines if you want to hardcode values for this runtime.
# api_key = "paste-your-TAMU_API_KEY-here"
# azure_endpoint = "https://your-resource.openai.azure.com/"
# api_version = "2024-12-01-preview"
# requested_model = "protected.gpt-5.2"
# deployment = "gpt-5.2-deep-learning-fundamentals"

if not api_key:
    api_key = getpass("Paste TAMU_API_KEY: ").strip()
if not azure_endpoint:
    azure_endpoint = input("Paste TAMU_AZURE_ENDPOINT: ").strip()
if not api_version:
    api_version = input("Paste TAMU_API_VERSION [2024-12-01-preview]: ").strip() or "2024-12-01-preview"
if not requested_model:
    requested_model = input("Paste OPENAI_MODEL [protected.gpt-5.2]: ").strip() or "protected.gpt-5.2"
if not deployment:
    deployment = input("Paste deployment [gpt-5.2-deep-learning-fundamentals]: ").strip() or deployment_aliases.get(requested_model, requested_model)

assert api_key, "Missing TAMU_API_KEY"
assert azure_endpoint, "Missing TAMU_AZURE_ENDPOINT"
assert api_version, "Missing TAMU_API_VERSION"
assert deployment, "Missing deployment name"

print("Manual override cell checked.")
print("Deployment that will be used:", deployment)


Manual override cell checked.
Deployment that will be used: gpt-5.2-deep-learning-fundamentals


## Connection Test

This is the actual API call. It uses the Azure OpenAI client directly, which is the path that worked in verification.


In [4]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint.rstrip("/"),
    api_version=api_version,
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "user", "content": 'Return a compact JSON object exactly like {"ok": true}.'}
    ],
    max_completion_tokens=80,
)

text = response.choices[0].message.content
usage = getattr(response, "usage", None)

print("Connection worked.")
print("Response model :", getattr(response, "model", deployment))
print("Text preview   :", text)
if usage is not None:
    print("Prompt tokens  :", getattr(usage, "prompt_tokens", None))
    print("Completion toks:", getattr(usage, "completion_tokens", None))
    print("Total tokens   :", getattr(usage, "total_tokens", None))


Connection worked.
Response model : gpt-5.2-2025-12-11
Text preview   : {"ok": true}
Prompt tokens  : 18
Completion toks: 9
Total tokens   : 27


## Reusable Minimal Example

Use this in future notebooks or scripts once the connection test works:

```python
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=TAMU_API_KEY,
    azure_endpoint=TAMU_AZURE_ENDPOINT.rstrip("/"),
    api_version="2024-12-01-preview",
)

response = client.chat.completions.create(
    model="gpt-5.2-deep-learning-fundamentals",
    messages=[{"role": "user", "content": "Hello"}],
    max_completion_tokens=80,
)

print(response.choices[0].message.content)
```


## If It Fails

- `401` or `403`: your key or TAMU access is wrong
- `404`: the endpoint or deployment name is wrong
- missing values in the summary cell: set Colab Secrets or use the manual override cell
- do not switch this notebook to `responses` mode unless your Azure/TAMU setup explicitly supports it
